In [ ]:
import torch
import matplotlib.pyplot as plt
from MonteCarlo import generate_heston_asian_mc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Eszköz: {device}")

# 1. Paraméterek (Pontosan egyezniük kell a PINN kódoddal!)
params = {'r': 0.05, 'kappa': 2.0, 'theta': 0.1, 'sigma': 0.3, 'rho': -0.7}
T_maturity = 1.0
K_val = 1.55  # A te skálázód alapján az A_K közepe: (0.1 + 3.0) / 2

# 2. Generálás
anchors = generate_heston_asian_mc(
    N_anchors=2000, 
    N_paths=20000, 
    N_steps=100, 
    params=params, 
    T=T_maturity, 
    K=K_val, 
    device=device
)

# 3. Mentés a merevlemezre (Ezt fogja majd beolvasni a PINN!)
torch.save(anchors, 'heston_anchors.pth')
print("✓ Adatok elmentve a 'heston_anchors.pth' fájlba!")

# 4. Vizualizáció
S = anchors['S0']
U = anchors['prices']
v = anchors['v0']

plt.figure(figsize=(8, 5))
scatter = plt.scatter(S, U, c=v, cmap='viridis', alpha=0.6, s=15)
plt.colorbar(scatter, label='Kezdeti variancia (v0)')
plt.title('Monte Carlo Horgonypontok (Heston Ázsiai Opció)')
plt.xlabel('Kezdeti Árfolyam (S0)')
plt.ylabel('Opció Ára (U)')
plt.grid(True, alpha=0.3)
plt.show()